In [7]:
!pip install chardet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.4/199.4 kB 615.8 kB/s eta 0:00:00a 0:00:01


In [8]:
from bs4 import BeautifulSoup
import re
import json
from html import unescape
import chardet

def extract_all_text_from_html(html_content):
    """
    Максимально полное извлечение текста из HTML документа
    """
    # Автоопределение кодировки
    detected_encoding = chardet.detect(html_content)['encoding']
    if detected_encoding:
        try:
            html_content = html_content.decode(detected_encoding)
        except:
            html_content = html_content.decode('utf-8', errors='ignore')
    
    # Используем lxml парсер - самый надежный для "плохого" HTML
    soup = BeautifulSoup(html_content, 'lxml')
    
    all_texts = []
    
    # 1. Извлекаем весь видимый текст
    visible_text = soup.get_text(separator='\n', strip=True)
    all_texts.append(visible_text)
    
    # 2. Извлекаем текст из таблиц (иногда get_text() теряет структуру таблиц)
    for table in soup.find_all('table'):
        table_text = extract_table_text(table)
        if table_text.strip():
            all_texts.append(f"ТАБЛИЦА:\n{table_text}")
    
    # 3. Извлекаем текст из атрибутов
    attribute_texts = extract_attribute_texts(soup)
    if attribute_texts:
        all_texts.append("\n".join(attribute_texts))
    
    # 4. Извлекаем текст из JavaScript и CSS (иногда там есть данные)
    script_texts = extract_script_and_style_texts(soup)
    if script_texts:
        all_texts.append("\n".join(script_texts))
    
    # 5. Извлекаем текст из комментариев
    comment_texts = extract_comments(soup)
    if comment_texts:
        all_texts.append("\n".join(comment_texts))
    
    # 6. Извлекаем текст из скрытых элементов
    hidden_texts = extract_hidden_texts(soup)
    if hidden_texts:
        all_texts.append("\n".join(hidden_texts))
    
    # 7. Извлекаем текст из SVG
    svg_texts = extract_svg_texts(soup)
    if svg_texts:
        all_texts.append("\n".join(svg_texts))
    
    # Объединяем весь текст
    full_text = "\n\n".join(filter(None, all_texts))
    
    # Декодируем HTML сущности
    full_text = unescape(full_text)
    
    # Очистка от лишних пробелов и пустых строк
    full_text = re.sub(r'\n\s*\n', '\n\n', full_text)
    full_text = re.sub(r'[ \t]+', ' ', full_text)
    
    return full_text.strip()

def extract_table_text(table):
    """Извлекает текст из таблицы с сохранением структуры"""
    rows = []
    for tr in table.find_all('tr'):
        cells = []
        for td in tr.find_all(['td', 'th']):
            cell_text = td.get_text(separator=' ', strip=True)
            if cell_text:
                cells.append(cell_text)
        if cells:
            rows.append(" | ".join(cells))
    return "\n".join(rows)

def extract_attribute_texts(soup):
    """Извлекает текст из атрибутов элементов"""
    attribute_texts = []
    
    # Атрибуты, которые могут содержать текст
    text_attributes = ['title', 'alt', 'value', 'placeholder', 'data-*']
    
    for tag in soup.find_all(True):
        for attr_name, attr_value in tag.attrs.items():
            # Проверяем все атрибуты на наличие текста
            if isinstance(attr_value, str) and len(attr_value) > 3:
                # Пропускаем технические атрибуты
                if attr_name in ['class', 'id', 'style', 'href', 'src', 'onclick']:
                    continue
                
                # Пропускаем очень короткие значения и URL
                if len(attr_value) < 5 or attr_value.startswith(('http', 'www', '#', 'javascript:')):
                    continue
                
                # Добавляем значимый текст из атрибутов
                attribute_texts.append(f"[{tag.name}.{attr_name}]: {attr_value}")
    
    return attribute_texts

def extract_script_and_style_texts(soup):
    """Извлекает текст из JavaScript и CSS"""
    texts = []
    
    # JavaScript
    for script in soup.find_all('script'):
        if script.string and len(script.string) > 20:
            # Удаляем JavaScript код, оставляем только строки
            js_strings = re.findall(r'(["\'])(.*?)\1', script.string)
            for _, string_value in js_strings:
                if len(string_value) > 10 and not re.match(r'^[a-zA-Z0-9_]+$', string_value):
                    texts.append(f"JS_STRING: {string_value}")
    
    # CSS
    for style in soup.find_all('style'):
        if style.string and len(style.string) > 20:
            # Ищем текстовые значения в CSS
            css_texts = re.findall(r'content:\s*["\'](.*?)["\']', style.string)
            for text_value in css_texts:
                if len(text_value) > 5:
                    texts.append(f"CSS_CONTENT: {text_value}")
    
    return texts

def extract_comments(soup):
    """Извлекает текст из HTML комментариев"""
    from bs4 import Comment
    comments = soup.find_all(string=lambda text: isinstance(text, Comment))
    return [f"COMMENT: {comment.strip()}" for comment in comments if len(comment.strip()) > 5]

def extract_hidden_texts(soup):
    """Извлекает текст из скрытых элементов"""
    hidden_texts = []
    
    # Скрытые элементы через CSS
    hidden_selectors = [
        {'style': 'display:none'},
        {'style': re.compile('display\\s*:\\s*none', re.I)},
        {'style': re.compile('visibility\\s*:\\s*hidden', re.I)},
        {'class': re.compile('hidden|invisible|sr-only', re.I)},
        {'hidden': True}
    ]
    
    for selector in hidden_selectors:
        for element in soup.find_all(attrs=selector):
            text = element.get_text(separator=' ', strip=True)
            if len(text) > 10:
                hidden_texts.append(f"HIDDEN_ELEMENT: {text}")
    
    return hidden_texts

def extract_svg_texts(soup):
    """Извлекает текст из SVG элементов"""
    svg_texts = []
    
    for svg in soup.find_all('svg'):
        for text_element in svg.find_all(['text', 'tspan', 'title', 'desc']):
            text = text_element.get_text(strip=True)
            if text and len(text) > 2:
                svg_texts.append(f"SVG_TEXT: {text}")
    
    return svg_texts

# Пример использования
def process_medical_report(html_file_path, output_file_path=None):
    """
    Основная функция обработки медицинского отчета
    """
    try:
        # Читаем файл с обработкой кодировки
        with open(html_file_path, 'rb') as f:
            html_content = f.read()
        
        # Извлекаем весь текст
        full_text = extract_all_text_from_html(html_content)
        
        # Сохраняем результат
        if output_file_path:
            with open(output_file_path, 'w', encoding='utf-8') as f:
                f.write(full_text)
        
        print(f"✅ Успешно извлечено {len(full_text)} символов из {html_file_path}")
        if output_file_path:
            print(f"💾 Результат сохранен в {output_file_path}")
        
        return full_text
    
    except Exception as e:
        print(f"❌ Ошибка при обработке {html_file_path}: {str(e)}")
        return None

# Для обработки папки с отчетами
def process_folder_with_reports(folder_path, output_folder=None):
    """
    Обработка всех HTML файлов в папке
    """
    import os
    import glob
    
    html_files = glob.glob(os.path.join(folder_path, "*.html")) + \
                glob.glob(os.path.join(folder_path, "*.htm"))
    
    print(f"📁 Найдено {len(html_files)} HTML файлов для обработки")
    
    results = {}
    for html_file in html_files:
        file_name = os.path.basename(html_file)
        output_file = None
        
        if output_folder:
            os.makedirs(output_folder, exist_ok=True)
            output_file = os.path.join(output_folder, f"{os.path.splitext(file_name)[0]}.txt")
        
        text = process_medical_report(html_file, output_file)
        if text:
            results[file_name] = text
    
    print(f"🎉 Обработано успешно: {len(results)} из {len(html_files)} файлов")
    return results

In [9]:
# Пример 1: Обработка одного файла
text = process_medical_report('../../data_raw/Histores_HTML/АбалмасовМВ8040_history_0.html', 'extracted_text.txt')

✅ Успешно извлечено 124550 символов из ../../data_raw/Histores_HTML/АбалмасовМВ8040_history_0.html
💾 Результат сохранен в extracted_text.txt


In [10]:
from bs4 import BeautifulSoup, NavigableString
import re
import chardet
from html import unescape
import json
from collections import defaultdict

def extract_structured_blocks_from_html(html_content):
    """
    Извлекает текст из HTML и разделяет на логические блоки с сохранением контекста
    """
    # Автоопределение кодировки
    detected_encoding = chardet.detect(html_content)['encoding']
    if detected_encoding:
        try:
            html_content = html_content.decode(detected_encoding)
        except:
            html_content = html_content.decode('utf-8', errors='ignore')
    
    soup = BeautifulSoup(html_content, 'lxml')
    
    # Словарь для хранения блоков
    structured_blocks = {
        'document_metadata': {
            'title': soup.title.string.strip() if soup.title else 'Без названия',
            'total_blocks': 0,
            'processing_info': {
                'parser': 'lxml',
                'encoding': detected_encoding or 'utf-8'
            }
        },
        'logical_blocks': [],
        'orphaned_text': '',  # Текст, который не попал ни в один блок
        'raw_text': ''        # Полный текст для резервного использования
    }
    
    # Шаг 1: Находим все потенциальные заголовки и разделители
    potential_headers = find_potential_headers(soup)
    
    # Шаг 2: Группируем контент по заголовкам
    blocks = group_content_by_headers(soup, potential_headers)
    
    # Шаг 3: Обрабатываем каждый блок
    for block in blocks:
        processed_block = process_block(block)
        if processed_block['content'].strip():
            structured_blocks['logical_blocks'].append(processed_block)
    
    # Шаг 4: Собираем оставшийся текст, который не попал в блоки
    orphaned_text = extract_orphaned_text(soup, blocks)
    if orphaned_text.strip():
        structured_blocks['orphaned_text'] = orphaned_text
    
    # Шаг 5: Формируем полный текст для резерва
    structured_blocks['raw_text'] = "\n\n".join([
        block['content'] for block in structured_blocks['logical_blocks']
    ]) + "\n\n" + structured_blocks['orphaned_text']
    
    structured_blocks['document_metadata']['total_blocks'] = len(structured_blocks['logical_blocks'])
    
    return structured_blocks

def find_potential_headers(soup):
    """
    Находит все элементы, которые могут быть заголовками разделов
    """
    potential_headers = []
    
    # 1. Стандартные HTML заголовки
    header_tags = soup.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6'])
    for header in header_tags:
        text = header.get_text(strip=True)
        if text and len(text) < 100:  # Исключаем очень длинные "заголовки"
            potential_headers.append({
                'element': header,
                'text': text,
                'level': int(header.name[1]),
                'tag': header.name,
                'confidence': 0.9,
                'source': 'html_header'
            })
    
    # 2. Жирный текст, похожий на заголовок
    bold_elements = soup.find_all(['b', 'strong'])
    for bold in bold_elements:
        text = bold.get_text(strip=True)
        if text and 3 < len(text) < 80 and is_header_candidate(text):
            parent = bold.find_parent()
            if parent and parent.name in ['div', 'p', 'span']:
                # Проверяем, что это первый элемент в родителе
                children = list(parent.children)
                if children and children[0] == bold:
                    potential_headers.append({
                        'element': bold,
                        'text': text,
                        'level': 3,  # Условный уровень
                        'tag': bold.name,
                        'confidence': 0.7,
                        'source': 'bold_text'
                    })
    
    # 3. Элементы с классами, содержащими "header", "title", "section"
    header_classes = soup.find_all(class_=re.compile('header|title|section|heading|zagolovok', re.I))
    for element in header_classes:
        text = element.get_text(strip=True)
        if text and 3 < len(text) < 100:
            potential_headers.append({
                'element': element,
                'text': text,
                'level': 2,
                'tag': element.name,
                'confidence': 0.8,
                'source': 'css_class'
            })
    
    # 4. Таблицы с заголовками в первой строке
    tables = soup.find_all('table')
    for table in tables:
        first_row = table.find('tr')
        if first_row:
            headers = [th.get_text(strip=True) for th in first_row.find_all(['th', 'td'])]
            if headers and all(header.strip() for header in headers) and len(headers) < 6:
                # Считаем первую строку заголовком таблицы
                table_title = " | ".join(headers)
                potential_headers.append({
                    'element': table,
                    'text': f"ТАБЛИЦА: {table_title}",
                    'level': 4,
                    'tag': 'table_header',
                    'confidence': 0.85,
                    'source': 'table_header'
                })
    
    # Сортируем по порядку в документе и уровню доверия
    potential_headers.sort(key=lambda x: (
        x['element'].sourceline or 999999,
        -x['confidence'],
        x['level']
    ))
    
    # Удаляем дубликаты и очень похожие заголовки
    unique_headers = []
    seen_texts = set()
    
    for header in potential_headers:
        clean_text = re.sub(r'\s+', ' ', header['text']).strip().lower()
        
        # Проверяем на дубликаты с небольшими вариациями
        is_duplicate = False
        for seen in seen_texts:
            if len(clean_text) > 5 and re.search(re.escape(seen), clean_text):
                is_duplicate = True
                break
        
        if not is_duplicate and clean_text not in seen_texts:
            seen_texts.add(clean_text)
            unique_headers.append(header)
    
    return unique_headers

def is_header_candidate(text):
    """Проверяет, похож ли текст на заголовок"""
    # Слишком короткий или длинный - не заголовок
    if len(text) < 3 or len(text) > 120:
        return False
    
    # Заглавные буквы в начале или все заглавные
    if text[0].isupper() or text.isupper():
        return True
    
    # Содержит ключевые слова для медицинских отчетов
    medical_keywords = [
        'анамнез', 'диагноз', 'жалобы', 'история', 'исследование', 'анализ',
        'заключение', 'лечение', 'назначение', 'пациент', 'врач', 'осмотр',
        'показатели', 'норма', 'результат', 'препарат', 'дозировка', 'схема',
        'прогноз', 'рекомендации', 'эпикриз'
    ]
    
    text_lower = text.lower()
    return any(keyword in text_lower for keyword in medical_keywords)

def group_content_by_headers(soup, headers):
    """
    Группирует контент по найденным заголовкам
    """
    blocks = []
    
    if not headers:
        # Если заголовков нет, берем весь документ как один блок
        return [{
            'header': None,
            'content_elements': [soup.body] if soup.body else [soup],
            'source_type': 'full_document'
        }]
    
    # Сортируем заголовки по позиции в документе
    headers.sort(key=lambda x: x['element'].sourceline or 999999)
    
    # Группируем контент между заголовками
    for i, header in enumerate(headers):
        current_block = {
            'header': header,
            'content_elements': [],
            'source_type': 'section'
        }
        
        current_element = header['element']
        next_header = headers[i + 1]['element'] if i + 1 < len(headers) else None
        
        # Собираем все элементы после текущего заголовка и до следующего
        for sibling in current_element.next_siblings:
            if next_header and sibling == next_header:
                break
            
            if isinstance(sibling, NavigableString):
                continue
            
            if hasattr(sibling, 'descendants'):
                # Проверяем, что элемент не является частью другого заголовка
                is_part_of_other_header = False
                for other_header in headers:
                    if other_header != header and other_header['element'] in sibling.descendants:
                        is_part_of_other_header = True
                        break
                
                if not is_part_of_other_header:
                    current_block['content_elements'].append(sibling)
        
        # Если есть содержимое или это последний блок, добавляем его
        if current_block['content_elements'] or i == len(headers) - 1:
            blocks.append(current_block)
    
    # Добавляем контент до первого заголовка
    first_header = headers[0]['element']
    pre_header_content = []
    
    for element in soup.body.children if soup.body else soup.children:
        if element == first_header:
            break
        if not isinstance(element, NavigableString):
            pre_header_content.append(element)
    
    if pre_header_content:
        blocks.insert(0, {
            'header': None,
            'content_elements': pre_header_content,
            'source_type': 'pre_header'
        })
    
    return blocks

def process_block(block):
    """
    Обрабатывает один логический блок
    """
    header_info = block['header']
    content_elements = block['content_elements']
    
    block_data = {
        'block_id': f"block_{id(header_info) if header_info else 'noheader'}",
        'header_text': header_info['text'] if header_info else 'Без заголовка',
        'header_confidence': header_info['confidence'] if header_info else 0.0,
        'header_source': header_info['source'] if header_info else 'none',
        'source_type': block['source_type'],
        'content': '',
        'raw_html_snippet': '',
        'metadata': {
            'element_count': len(content_elements),
            'has_tables': any(elem.name == 'table' for elem in content_elements if hasattr(elem, 'name'))
        }
    }
    
    # Извлекаем содержимое блока
    block_content_parts = []
    
    for element in content_elements:
        if hasattr(element, 'get_text'):
            # Текст из элемента
            text = element.get_text(separator='\n', strip=True)
            if text.strip():
                block_content_parts.append(text)
            
            # Особая обработка таблиц
            if element.name == 'table':
                table_text = extract_table_text_simple(element)
                if table_text.strip():
                    block_content_parts.append(f"ТАБЛИЦА:\n{table_text}")
    
    block_data['content'] = "\n\n".join(block_content_parts).strip()
    
    # Формируем сниппет HTML для отладки
    if content_elements:
        first_element = content_elements[0]
        block_data['raw_html_snippet'] = str(first_element)[:200] + '...' if str(first_element) else ''
    
    # Автоматическая классификация блока по заголовку
    block_data['suggested_category'] = classify_block_by_header(block_data['header_text'])
    
    return block_data

def extract_table_text_simple(table):
    """Простое извлечение текста из таблицы"""
    rows = []
    for tr in table.find_all('tr'):
        cells = [td.get_text(strip=True) for td in tr.find_all(['td', 'th'])]
        if any(cells):
            rows.append(" | ".join(cell for cell in cells if cell))
    return "\n".join(rows)

def classify_block_by_header(header_text):
    """Автоматическая классификация блока по заголовку"""
    header_lower = header_text.lower()
    
    # Словарь категорий
    categories = {
        'patient_info': ['пациент', 'фамилия', 'имя', 'отчество', 'дата рождения', 'пол', 'адрес', 'паспорт', 'полис'],
        'complaints': ['жалобы', 'причины обращения', 'основные жалобы'],
        'history': ['анамнез', 'история заболевания', 'сопутствующие заболевания', 'наследственность', 'аллергии'],
        'examination': ['осмотр', 'состояние', 'объективный статус', 'данные осмотра'],
        'diagnosis': ['диагноз', 'предварительный диагноз', 'клинический диагноз', 'основной диагноз'],
        'tests': ['анализы', 'исследования', 'лабораторные', 'инструментальные', 'показатели', 'результаты'],
        'treatment': ['лечение', 'назначение', 'препараты', 'дозировка', 'схема', 'рекомендации'],
        'conclusion': ['заключение', 'эпикриз', 'прогноз', 'план ведения']
    }
    
    for category, keywords in categories.items():
        if any(keyword in header_lower for keyword in keywords):
            return category
    return 'other'

def extract_orphaned_text(soup, blocks):
    """
    Извлекает текст, который не попал ни в один блок
    """
    # Собираем все элементы, которые были использованы в блоках
    used_elements = set()
    for block in blocks:
        if block['header']:
            used_elements.add(block['header']['element'])
        used_elements.update(block['content_elements'])
    
    # Ищем текст в элементах, которые не были использованы
    orphaned_parts = []
    
    for element in soup.find_all(True):
        if element not in used_elements:
            text = element.get_text(strip=True)
            if text and len(text) > 10:  # Игнорируем очень короткий текст
                # Проверяем, что это не технический текст
                if not re.match(r'^(cookie|javascript|css|meta|script|style)', element.name, re.I):
                    orphaned_parts.append(f"[{element.name}]: {text}")
    
    return "\n\n".join(orphaned_parts)

# Пример использования
def process_medical_report_structured(html_file_path, output_json_path=None, output_txt_path=None):
    """
    Обработка HTML отчета с сохранением структуры
    """
    try:
        # Читаем файл
        with open(html_file_path, 'rb') as f:
            html_content = f.read()
        
        # Извлекаем структурированные данные
        structured_data = extract_structured_blocks_from_html(html_content)
        
        # Сохраняем в JSON для анализа структуры
        if output_json_path:
            with open(output_json_path, 'w', encoding='utf-8') as f:
                json.dump(structured_data, f, ensure_ascii=False, indent=2)
        
        # Создаем человекочитаемый текстовый файл с разметкой блоков
        if output_txt_path:
            with open(output_txt_path, 'w', encoding='utf-8') as f:
                f.write(f"= ДОКУМЕНТ: {structured_data['document_metadata']['title']} =\n")
                f.write(f"= ВСЕГО БЛОКОВ: {structured_data['document_metadata']['total_blocks']} =\n\n")
                
                for i, block in enumerate(structured_data['logical_blocks'], 1):
                    f.write(f"{'='*80}\n")
                    f.write(f"БЛОК #{i}\n")
                    f.write(f"Заголовок: {block['header_text']}\n")
                    f.write(f"Категория: {block['suggested_category']}\n")
                    f.write(f"Уверенность: {block['header_confidence']:.2f}\n")
                    f.write(f"Источник: {block['header_source']}\n")
                    f.write(f"Тип: {block['source_type']}\n")
                    f.write(f"{'-'*80}\n")
                    f.write(block['content'] + "\n\n")
                
                if structured_data['orphaned_text'].strip():
                    f.write(f"{'='*80}\n")
                    f.write("НЕПРИВЯЗАННЫЙ ТЕКСТ (вне блоков):\n")
                    f.write(f"{'-'*80}\n")
                    f.write(structured_data['orphaned_text'] + "\n\n")
        
        print(f"✅ Успешно извлечено {structured_data['document_metadata']['total_blocks']} блоков из {html_file_path}")
        if output_json_path:
            print(f"💾 Структура сохранена в {output_json_path}")
        if output_txt_path:
            print(f"💾 Человекочитаемый вариант сохранен в {output_txt_path}")
        
        return structured_data
    
    except Exception as e:
        print(f"❌ Ошибка при обработке {html_file_path}: {str(e)}")
        return None

# Функция для подготовки данных к обучению моделей
def prepare_training_data(structured_data, include_metadata=False):
    """
    Подготавливает данные из структурированного формата для обучения моделей
    """
    training_samples = []
    
    for block in structured_data['logical_blocks']:
        sample = {
            'text': block['content'],
            'block_id': block['block_id'],
            'category': block['suggested_category'],
            'header': block['header_text'],
            'confidence': block['header_confidence']
        }
        
        if include_metadata:
            sample.update({
                'metadata': {
                    'source_type': block['source_type'],
                    'header_source': block['header_source'],
                    'has_tables': block['metadata']['has_tables']
                }
            })
        
        training_samples.append(sample)
    
    # Добавляем непривязанный текст как отдельные сэмплы
    orphaned_text = structured_data['orphaned_text'].strip()
    if orphaned_text:
        # Разбиваем большой блок на части по логическим разделам
        orphaned_parts = re.split(r'\n{2,}', orphaned_text)
        for i, part in enumerate(orphaned_parts):
            if len(part.strip()) > 20:
                training_samples.append({
                    'text': part,
                    'block_id': f"orphaned_{i}",
                    'category': 'orphaned',
                    'header': 'Непривязанный текст',
                    'confidence': 0.0
                })
    
    return training_samples

In [11]:
# Обработка одного файла
structured_data = process_medical_report_structured(
    '../../data_raw/Histores_HTML/АбалмасовМВ8040_history_0.html',
    'structured_data.json',    # Полная структура для анализа
    'human_readable.txt'       # Человекочитаемая версия
)

# Подготовка данных для обучения логистической регрессии
training_samples = prepare_training_data(structured_data)

✅ Успешно извлечено 27 блоков из ../../data_raw/Histores_HTML/АбалмасовМВ8040_history_0.html
💾 Структура сохранена в structured_data.json
💾 Человекочитаемый вариант сохранен в human_readable.txt


In [14]:
import re
import json
import os
import glob
from bs4 import BeautifulSoup, NavigableString
from html import unescape
import chardet
import pandas as pd
from collections import defaultdict
import numpy as np

def parse_doca_plus_report(html_content):
    """
    Специализированный парсер для отчетов системы DocA+
    Полностью самодостаточный и рабочий код
    """
    # Автоопределение кодировки
    detected = chardet.detect(html_content)
    encoding = detected['encoding'] or 'utf-8'
    
    try:
        html_text = html_content.decode(encoding)
    except (UnicodeDecodeError, LookupError):
        try:
            html_text = html_content.decode('utf-8', errors='ignore')
        except:
            html_text = html_content.decode('cp1251', errors='ignore')
    
    # Создаем объект BeautifulSoup
    soup = BeautifulSoup(html_text, 'lxml')
    
    # Основная структура данных
    result = {
        'metadata': {
            'system': 'DocA+ Clinical Information System',
            'patient_id': 'unknown',
            'creation_date': 'unknown',
            'department': 'unknown',
            'report_title': soup.title.string.strip() if soup.title else 'Без названия'
        },
        'clinical_sections': {
            'patient_info': [],
            'complaints': [],
            'history': [],
            'physical_exam': [],
            'diagnosis': [],
            'lab_results': [],
            'instrumental': [],
            'treatment': [],
            'medications': [],
            'conclusion': [],
            'recommendations': []
        },
        'tables': [],
        'raw_text_blocks': [],
        'quality_metrics': {
            'total_text_length': 0,
            'section_coverage': 0.0,
            'confidence_score': 0.0
        }
    }
    
    # Извлечение метаданных
    result['metadata']['patient_id'] = _extract_patient_id(soup)
    result['metadata']['creation_date'] = _extract_creation_date(soup)
    result['metadata']['department'] = _extract_department(soup)
    
    # Обнаружение клинических разделов
    _detect_doca_sections(soup, result)
    
    # Извлечение таблиц
    _extract_doca_tables(soup, result)
    
    # Сбор всех текстовых блоков
    _extract_all_text_blocks(soup, result)
    
    # Расчет метрик качества
    _calculate_quality_metrics(result)
    
    return result

def _extract_patient_id(soup):
    """Внутренняя функция для извлечения ID пациента"""
    patterns = [
        r'ID пациента[:\s]*(\d+)',
        r'Пациент[:\s]*(\d+)',
        r'№ истории[:\s]*(\d+)',
        r'patient_id[:\s]*(\d+)',
        r'record_number[:\s]*(\d+)'
    ]
    
    text = soup.get_text()
    for pattern in patterns:
        match = re.search(pattern, text, re.I)
        if match:
            return match.group(1)
    
    return 'unknown'

def _extract_creation_date(soup):
    """Внутренняя функция для извлечения даты создания"""
    date_patterns = [
        r'\d{2}[./-]\d{2}[./-]\d{4}',
        r'\d{4}[./-]\d{2}[./-]\d{2}',
        r'\d{1,2}\s+(?:января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября|декабря)\s+\d{4}'
    ]
    
    for pattern in date_patterns:
        matches = re.findall(pattern, soup.get_text())
        if matches:
            return matches[0]
    
    return 'unknown'

def _extract_department(soup):
    """Внутренняя функция для извлечения отделения"""
    department_keywords = ['отделение', 'терапевтическое', 'хирургическое', 'кардиологическое', 'неврологическое', 'пульмонологическое']
    text = soup.get_text().lower()
    
    for keyword in department_keywords:
        if keyword in text:
            start_idx = text.find(keyword)
            end_idx = min(start_idx + 50, len(text))
            context = text[start_idx:end_idx].split('\n')[0].strip()
            return context.capitalize()
    
    return 'unknown'

def _detect_doca_sections(soup, result):
    """Внутренняя функция для обнаружения клинических разделов"""
    # Словарь заголовков DocA+
    doca_headers = {
        'patient_info': ['пациент', 'больной', 'личные данные', 'паспортные данные', 'адрес', 'фио', 'фамилия имя отчество'],
        'complaints': ['жалобы', 'причины обращения', 'основные жалобы', 'анамнез настоящего заболевания', 'жалобы при поступлении'],
        'history': ['анамнез', 'история заболевания', 'сопутствующие заболевания', 'аллергический анамнез', 'наследственность', 'социальный анамнез'],
        'physical_exam': ['осмотр', 'состояние пациента', 'физикальный осмотр', 'объективный статус', 'vital signs', 'температура', 'пульс', 'давление'],
        'diagnosis': ['диагноз', 'клинический диагноз', 'основной диагноз', 'сопутствующий диагноз', 'предварительный диагноз', 'заключительный диагноз'],
        'lab_results': ['анализы', 'лабораторные исследования', 'показатели крови', 'биохимия', 'клинический анализ крови', 'общий анализ мочи'],
        'instrumental': ['инструментальные исследования', 'рентген', 'узи', 'мрт', 'кт', 'эхокг', 'эгдс', 'фгдс', 'ультразвук'],
        'treatment': ['лечение', 'назначения', 'терапия', 'схема лечения', 'план лечения', 'проведенное лечение'],
        'medications': ['медикаменты', 'препараты', 'дозировка', 'лекарства', 'схема приема', 'антибиотики', 'анальгетики'],
        'conclusion': ['заключение', 'эпикриз', 'итоги', 'резюме', 'клиническое заключение', 'заключительный диагноз'],
        'recommendations': ['рекомендации', 'план ведения', 'диспансеризация', 'наблюдение', 'выписка', 'амбулаторное лечение']
    }
    
    # Поиск заголовков в документе
    for header_tag in ['h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'strong', 'b', 'span']:
        for header in soup.find_all(header_tag):
            header_text = header.get_text(strip=True).lower()
            if 3 < len(header_text) < 150:  # Оптимальная длина заголовка
                
                # Ищем, к какому разделу относится заголовок
                for section, keywords in doca_headers.items():
                    for keyword in keywords:
                        if keyword in header_text:
                            # Извлекаем содержимое после заголовка
                            content = _extract_content_after_header(header)
                            if len(content) > 20:  # Минимальная длина содержимого
                                result['clinical_sections'][section].append({
                                    'header': header.get_text(strip=True),
                                    'content': content,
                                    'confidence': 0.9 if header.name in ['h1', 'h2', 'h3'] else 0.7,
                                    'source': f'html_{header.name}'
                                })
                            break
    
    # Поиск по ключевым словам в тексте (если не нашли через заголовки)
    full_text = soup.get_text()
    sentences = [s.strip() for s in re.split(r'[.!?;]\s+', full_text) if s.strip()]
    
    for section, keywords in doca_headers.items():
        if not result['clinical_sections'][section]:  # Если раздел еще пустой
            for sentence in sentences:
                sentence_lower = sentence.lower()
                if len(sentence) > 30 and any(keyword in sentence_lower for keyword in keywords):
                    # Ищем контекст вокруг предложения
                    context = _extract_context_around_text(sentence, full_text)
                    if len(context) > 50:
                        result['clinical_sections'][section].append({
                            'header': 'Автоматически определенный раздел',
                            'content': context,
                            'confidence': 0.5,
                            'source': 'text_analysis'
                        })
                        break

def _extract_content_after_header(header_element, max_chars=2000):
    """Внутренняя функция для извлечения содержимого после заголовка"""
    content_parts = []
    current = header_element
    
    while current and len(" ".join(content_parts)) < max_chars:
        next_sibling = current.next_sibling
        
        if next_sibling is None:
            break
        
        if isinstance(next_sibling, NavigableString):
            text = next_sibling.strip()
            if text and len(text) > 2:
                content_parts.append(text)
        
        elif hasattr(next_sibling, 'name'):
            # Останавливаемся на следующем заголовке или таблице
            if next_sibling.name in ['h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'table', 'hr']:
                break
            
            text = next_sibling.get_text(strip=True)
            if text and len(text) > 5:
                content_parts.append(text)
        
        current = next_sibling
    
    return " ".join(content_parts)[:max_chars].strip()

def _extract_context_around_text(target_text, full_text, context_size=200):
    """Внутренняя функция для извлечения контекста вокруг текста"""
    start_idx = full_text.lower().find(target_text.lower())
    if start_idx == -1:
        return target_text
    
    context_start = max(0, start_idx - context_size)
    context_end = min(len(full_text), start_idx + len(target_text) + context_size)
    
    return full_text[context_start:context_end].strip()

def _extract_doca_tables(soup, result):
    """Внутренняя функция для извлечения таблиц"""
    for i, table in enumerate(soup.find_all('table')):
        # Извлечение заголовков таблицы
        headers = []
        header_row = table.find('tr')
        if header_row:
            for th in header_row.find_all(['th', 'td']):
                header_text = th.get_text(strip=True)
                if header_text:
                    headers.append(header_text)
        
        # Извлечение данных таблицы
        rows = []
        for tr in table.find_all('tr')[1:]:  # Пропускаем заголовок
            cells = []
            for td in tr.find_all(['td', 'th']):
                cell_text = td.get_text(strip=True)
                cells.append(cell_text)
            if any(cells):  # Если есть хотя бы одна непустая ячейка
                rows.append(cells)
        
        if rows:  # Если таблица не пустая
            # Определение типа таблицы по заголовкам
            table_text = " ".join(headers + [cell for row in rows for cell in row]).lower()
            suggested_section = 'unknown'
            
            if any(word in table_text for word in ['анализ', 'показатель', 'норма', 'результат', 'кровь', 'моча']):
                suggested_section = 'lab_results'
            elif any(word in table_text for word in ['диагноз', 'мкб', 'код', 'ds']):
                suggested_section = 'diagnosis'
            elif any(word in table_text for word in ['препарат', 'доза', 'прием', 'схема', 'таблетка']):
                suggested_section = 'medications'
            elif any(word in table_text for word in ['осмотр', 'состояние', 'параметр', 'температура', 'пульс', 'давление']):
                suggested_section = 'physical_exam'
            
            # Формирование таблицы
            table_data = {
                'table_id': f"table_{i}",
                'headers': headers,
                'rows': rows,
                'suggested_section': suggested_section,
                'row_count': len(rows),
                'column_count': len(headers) if headers else (len(rows[0]) if rows else 0)
            }
            result['tables'].append(table_data)

def _extract_all_text_blocks(soup, result):
    """Внутренняя функция для сбора всех текстовых блоков"""
    # Извлекаем весь видимый текст
    visible_text = soup.get_text(separator='\n', strip=True)
    paragraphs = [p.strip() for p in visible_text.split('\n') if p.strip() and 30 < len(p.strip()) < 1000]
    
    for i, paragraph in enumerate(paragraphs[:20]):  # Ограничиваем количество для производительности
        result['raw_text_blocks'].append({
            'block_id': f"block_{i}",
            'content': paragraph,
            'length': len(paragraph)
        })

def _calculate_quality_metrics(result):
    """Внутренняя функция для расчета метрик качества"""
    # Подсчет заполненных разделов
    total_sections = len(result['clinical_sections'])
    filled_sections = sum(1 for blocks in result['clinical_sections'].values() if blocks)
    
    result['quality_metrics']['section_coverage'] = filled_sections / total_sections if total_sections > 0 else 0
    
    # Подсчет общей длины текста
    total_text_length = 0
    for blocks in result['clinical_sections'].values():
        for block in blocks:
            total_text_length += len(block['content'])
    
    for block in result['raw_text_blocks']:
        total_text_length += len(block['content'])
    
    for table in result['tables']:
        for row in table['rows']:
            for cell in row:
                total_text_length += len(str(cell))
    
    result['quality_metrics']['total_text_length'] = total_text_length
    result['quality_metrics']['confidence_score'] = min(1.0, total_text_length / 5000)  # Нормализация

def prepare_training_data(parsed_report):
    """
    Подготовка данных из распарсенного отчета для обучения моделей
    """
    training_samples = []
    
    # Обработка клинических разделов
    for section_name, blocks in parsed_report['clinical_sections'].items():
        for block in blocks:
            if len(block['content']) > 50:  # Минимальная длина для обучения
                training_samples.append({
                    'text': block['content'],
                    'section': section_name,
                    'confidence': block['confidence'],
                    'source': block['source'],
                    'patient_id': parsed_report['metadata']['patient_id'],
                    'document_type': 'doca_clinical_report'
                })
    
    # Обработка таблиц
    for table in parsed_report['tables']:
        # Преобразуем таблицу в текстовый формат
        table_text = ""
        if table['headers']:
            table_text += " | ".join(table['headers']) + "\n"
        
        for row in table['rows']:
            table_text += " | ".join(str(cell) for cell in row) + "\n"
        
        if len(table_text) > 100 and table['suggested_section'] != 'unknown':
            training_samples.append({
                'text': f"ТАБЛИЦА ({table['suggested_section']}):\n{table_text}",
                'section': table['suggested_section'],
                'confidence': 0.8,
                'source': 'table',
                'patient_id': parsed_report['metadata']['patient_id'],
                'document_type': 'doca_clinical_report'
            })
    
    # Обработка необработанных текстовых блоков
    for block in parsed_report['raw_text_blocks']:
        if 100 < len(block['content']) < 2000:  # Оптимальный размер для обучения
            # Пытаемся определить секцию по содержимому
            section = _guess_section_from_text(block['content'])
            training_samples.append({
                'text': block['content'],
                'section': section,
                'confidence': 0.3 if section == 'other' else 0.5,
                'source': 'raw_text',
                'patient_id': parsed_report['metadata']['patient_id'],
                'document_type': 'doca_clinical_report'
            })
    
    return training_samples

def _guess_section_from_text(text):
    """Внутренняя функция для угадывания секции по тексту"""
    text_lower = text.lower()
    
    if any(word in text_lower for word in ['жалоб', 'жалуется', 'обращается с']):
        return 'complaints'
    elif any(word in text_lower for word in ['анамнез', 'ранее болел', 'в прошлом']):
        return 'history'
    elif any(word in text_lower for word in ['диагноз', 'поставлен диагноз', 'клинический диагноз']):
        return 'diagnosis'
    elif any(word in text_lower for word in ['анализы', 'показатели', 'результаты', 'лабораторные']):
        return 'lab_results'
    elif any(word in text_lower for word in ['лечение', 'назначено', 'препарат', 'схема', 'дозировка']):
        return 'treatment'
    elif any(word in text_lower for word in ['заключение', 'эпикриз', 'прогноз', 'рекомендовано']):
        return 'conclusion'
    
    return 'other'

def process_doca_reports_folder(input_folder, output_folder):
    """
    Обработка всех HTML файлов в папке и создание единого датасета
    """
    os.makedirs(output_folder, exist_ok=True)
    
    # Поиск HTML файлов
    html_files = glob.glob(os.path.join(input_folder, "*.html")) + \
                glob.glob(os.path.join(input_folder, "*.htm"))
    
    print(f"📁 Найдено {len(html_files)} HTML файлов для обработки")
    
    all_training_samples = []
    processing_stats = {
        'total_files': len(html_files),
        'successfully_processed': 0,
        'failed_files': [],
        'section_counts': defaultdict(int)
    }
    
    # Обработка каждого файла
    for i, html_file in enumerate(html_files, 1):
        file_name = os.path.basename(html_file)
        print(f"📋 Обработка файла {i}/{len(html_files)}: {file_name}")
        
        try:
            with open(html_file, 'rb') as f:
                html_content = f.read()
            
            # Парсинг отчета
            parsed_report = parse_doca_plus_report(html_content)
            
            # Подготовка данных для обучения
            training_samples = prepare_training_data(parsed_report)
            
            # Обновление статистики
            processing_stats['successfully_processed'] += 1
            for sample in training_samples:
                processing_stats['section_counts'][sample['section']] += 1
            
            # Добавление в общий датасет
            all_training_samples.extend(training_samples)
            
            # Сохранение промежуточных результатов
            json_output = os.path.join(output_folder, f"{os.path.splitext(file_name)[0]}.json")
            with open(json_output, 'w', encoding='utf-8') as f:
                json.dump(parsed_report, f, ensure_ascii=False, indent=2)
            
        except Exception as e:
            print(f"❌ Ошибка при обработке {file_name}: {str(e)}")
            processing_stats['failed_files'].append({
                'file': file_name,
                'error': str(e)
            })
    
    # Создание единого датасета
    if all_training_samples:
        df = pd.DataFrame(all_training_samples)
        
        # Сохранение CSV
        csv_path = os.path.join(output_folder, "unified_doca_dataset.csv")
        df.to_csv(csv_path, index=False, encoding='utf-8')
        
        # Создание отчета
        report = _generate_processing_report(processing_stats, df)
        report_path = os.path.join(output_folder, "processing_report.txt")
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write(report)
        
        print(f"\n✅ Обработка завершена!")
        print(f"💾 Датасет сохранен: {csv_path}")
        print(f"📊 Отчет сохранен: {report_path}")
        print(f"📈 Всего сэмплов: {len(df)}")
        
        return df, processing_stats, report
    
    print("❌ Не удалось создать датасет - нет обработанных данных")
    return None, processing_stats, None

def _generate_processing_report(stats, df):
    """Генерация подробного отчета об обработке"""
    report_lines = []
    
    report_lines.append("=" * 60)
    report_lines.append("ОТЧЕТ ОБ ОБРАБОТКЕ ОТЧЕТОВ DOCA+")
    report_lines.append("=" * 60)
    report_lines.append("")
    
    # Общая статистика
    report_lines.append("📊 ОБЩАЯ СТАТИСТИКА:")
    report_lines.append(f"  • Всего файлов: {stats['total_files']}")
    report_lines.append(f"  • Успешно обработано: {stats['successfully_processed']}")
    report_lines.append(f"  • Ошибок обработки: {len(stats['failed_files'])}")
    report_lines.append(f"  • Всего текстовых сэмплов: {len(df) if df is not None else 0}")
    report_lines.append("")
    
    # Статистика по секциям
    report_lines.append("🏷️ СТАТИСТИКА ПО СЕКЦИЯМ:")
    for section, count in sorted(stats['section_counts'].items()):
        percentage = (count / len(df)) * 100 if len(df) > 0 else 0
        report_lines.append(f"  • {section}: {count} сэмплов ({percentage:.1f}%)")
    report_lines.append("")
    
    # Статистика по длине текста
    if df is not None:
        df['text_length'] = df['text'].apply(len)
        report_lines.append("📏 СТАТИСТИКА ПО ДЛИНЕ ТЕКСТА:")
        report_lines.append(f"  • Минимальная длина: {df['text_length'].min()} символов")
        report_lines.append(f"  • Средняя длина: {df['text_length'].mean():.0f} символов")
        report_lines.append(f"  • Максимальная длина: {df['text_length'].max()} символов")
        report_lines.append(f"  • Медиана длины: {df['text_length'].median():.0f} символов")
        report_lines.append("")
    
    # Примеры данных
    report_lines.append("🔍 ПРИМЕРЫ ДАННЫХ:")
    if df is not None:
        sample_size = min(5, len(df))
        samples = df.sample(sample_size)
        for i, (_, row) in enumerate(samples.iterrows(), 1):
            text_preview = row['text'][:150] + '...' if len(row['text']) > 150 else row['text']
            report_lines.append(f"\nПример {i}:")
            report_lines.append(f"  Секция: {row['section']}")
            report_lines.append(f"  Источник: {row['source']}")
            report_lines.append(f"  Текст: {text_preview}")
    
    return "\n".join(report_lines)




In [15]:
with open('../../data_raw/Histores_HTML/АбалмасовМВ8040_history_0.html', 'rb') as f:
    html_content = f.read()

# Укажите пути к вашим папкам
INPUT_FOLDER = "."  # Папка с HTML отчетами DocA+
OUTPUT_FOLDER = "."    # Папка для результатов

# Запуск обработки
print("🚀 ЗАПУСК ОБРАБОТКИ ОТЧЕТОВ DOCA+")
print("=" * 50)

dataset_df, stats, report = process_doca_reports_folder(INPUT_FOLDER, OUTPUT_FOLDER)

🚀 ЗАПУСК ОБРАБОТКИ ОТЧЕТОВ DOCA+
📁 Найдено 1 HTML файлов для обработки
📋 Обработка файла 1/1: АбашкинЮН_history_0.html

✅ Обработка завершена!
💾 Датасет сохранен: ./unified_doca_dataset.csv
📊 Отчет сохранен: ./processing_report.txt
📈 Всего сэмплов: 32


In [16]:
test_data = pd.read_csv("./unified_doca_dataset.csv")

In [17]:
test_data

,text,section,confidence,source,patient_id,document_type
0,Дата обязательной явки к врачу в поликлинику п...,patient_info,0.7,html_h4,14952,doca_clinical_report
1,Абашкин Ю. Н. 60 лет От...,patient_info,0.7,html_b,14952,doca_clinical_report
2,Страховая организация: .Восточно-страховой Аль...,patient_info,0.7,html_b,14952,doca_clinical_report
3,"Приморский край, ШкотовскийЗАТО Большой Камень...",patient_info,0.7,html_b,14952,doca_clinical_report
4,ачебный осмотр Абашкин Юрий Николаевич Возрас...,complaints,0.5,text_analysis,14952,doca_clinical_report
5,"ИБ 14952 Дата и время осмотра 20.09.2016, ...",history,0.5,text_analysis,14952,doca_clinical_report
6,осле выписки: Исход заболевания: выписан с у...,physical_exam,0.5,text_analysis,14952,doca_clinical_report
7,п/о: ИБС. Нестабильная стенокардия высокой сте...,diagnosis,0.7,html_b,14952,doca_clinical_report
8,п/о: ИБС. Нестабильная стенокардия высокой сте...,diagnosis,0.7,html_b,14952,doca_clinical_report
9,п/о: ИБС. Нестабильная стенокардия высокой сте...,diagnosis,0.7,html_b,14952,doca_clinical_report


In [18]:
test_data['text']

0     Дата обязательной явки к врачу в поликлинику п...
1     Абашкин                        Ю. Н. 60 лет От...
2     Страховая организация: .Восточно-страховой Аль...
3     Приморский край, ШкотовскийЗАТО Большой Камень...
4     ачебный осмотр Абашкин Юрий Николаевич  Возрас...
5     ИБ 14952   Дата и время осмотра 20.09.2016,   ...
6     осле выписки:   Исход заболевания: выписан с у...
7     п/о: ИБС. Нестабильная стенокардия высокой сте...
8     п/о: ИБС. Нестабильная стенокардия высокой сте...
9     п/о: ИБС. Нестабильная стенокардия высокой сте...
10    п/о: ИБС. Нестабильная стенокардия высокой сте...
11    Аортауплотнена;кальциноз ст.Фибриозное кольцо ...
12    билирубин (BIL)=Не обнаружено ; уробилинон (UR...
13    Ф.И.О.Абашкин Юрий Николаевич;    Возраст 60 л...
14    Агрегация с индукторами (АДФ) (Платно)=21.4 % ...
15    ентированием ПКА. Гипертоническая болезнь 3 ст...
16    ого риска.  .Консультирован с РСЦ . Переведен ...
17    Дата перевода: Время перевода: Ф.И.О.: Аба

In [20]:
import re
import json
import os
import glob
from bs4 import BeautifulSoup, NavigableString, Comment
from html import unescape
import chardet
import pandas as pd
from collections import defaultdict
import numpy as np
import textwrap

def extract_all_text_with_structure(html_content):
    """
    МАКСИМАЛЬНО ПОЛНЫЙ ПАРСЕР - извлекает ВЕСЬ текст со ВСЕМИ деталями и структурой
    Никакой фильтрации, никаких потерь данных
    """
    # Автоопределение кодировки - более надежный подход
    detected = chardet.detect(html_content)
    encoding = detected['encoding'] or 'utf-8'
    
    try:
        html_text = html_content.decode(encoding)
    except (UnicodeDecodeError, LookupError, TypeError):
        try:
            html_text = html_content.decode('utf-8', errors='ignore')
        except:
            try:
                html_text = html_content.decode('cp1251', errors='ignore')
            except:
                html_text = str(html_content)
    
    # Создаем объект BeautifulSoup с максимальной детализацией
    soup = BeautifulSoup(html_text, 'lxml')
    
    # Основная структура - НИКАКОЙ ФИЛЬТРАЦИИ!
    result = {
        'document_metadata': {
            'title': soup.title.string.strip() if soup.title else 'Без названия',
            'total_text_length': 0,
            'total_elements_processed': 0,
            'encoding_used': encoding,
            'parser_used': 'lxml'
        },
        'detailed_blocks': [],  # КАЖДЫЙ текстовый блок как отдельная запись
        'raw_full_text': '',    # Полный текст без структуры (для резерва)
        'element_hierarchy': [], # Иерархия элементов
        'processing_stats': {
            'headers_found': 0,
            'paragraphs_found': 0,
            'tables_found': 0,
            'lists_found': 0,
            'other_elements_found': 0,
            'text_blocks_created': 0
        }
    }
    
    # Шаг 1: Извлекаем полный текст для резерва (никакой фильтрации)
    full_text = soup.get_text(separator='\n', strip=False)
    full_text = unescape(full_text)  # Декодируем HTML сущности
    full_text = re.sub(r'\n{3,}', '\n\n', full_text)  # Нормализуем переносы строк
    full_text = re.sub(r'[ \t]{2,}', ' ', full_text)  # Нормализуем пробелы
    result['raw_full_text'] = full_text.strip()
    
    # Шаг 2: Извлекаем КАЖДЫЙ текстовый блок с максимальной детализацией
    _extract_all_text_blocks_detailed(soup, result)
    
    # Шаг 3: Извлекаем ВСЕ таблицы без фильтрации
    _extract_all_tables_completely(soup, result)
    
    # Шаг 4: Извлекаем текст из атрибутов (title, alt, data-*)
    _extract_text_from_attributes(soup, result)
    
    # Шаг 5: Извлекаем комментарии (иногда там есть данные)
    _extract_html_comments(soup, result)
    
    # Шаг 6: Извлекаем скрытый текст (display:none, visibility:hidden)
    _extract_hidden_text(soup, result)
    
    # Шаг 7: Строим иерархию элементов
    _build_element_hierarchy(soup, result)
    
    # Шаг 8: Подсчет статистики
    result['document_metadata']['total_text_length'] = len(result['raw_full_text'])
    result['processing_stats']['text_blocks_created'] = len(result['detailed_blocks'])
    
    return result

def _extract_all_text_blocks_detailed(soup, result):
    """
    Извлекает КАЖДЫЙ текстовый блок без ФИЛЬТРАЦИИ
    Обрабатывает ВСЕ элементы, а не только определенные теги
    """
    block_counter = 0
    
    # Рекурсивная функция для обхода ВСЕХ элементов
    def process_element(element, parent_path="", depth=0):
        nonlocal block_counter
        
        if isinstance(element, NavigableString):
            text = str(element).strip()
            if text and not text.isspace():
                block_counter += 1
                block_id = f"block_{block_counter}"
                
                result['detailed_blocks'].append({
                    'block_id': block_id,
                    'text': text,
                    'element_type': 'text_node',
                    'parent_path': parent_path,
                    'depth': depth,
                    'position': block_counter,
                    'raw_html': str(element),
                    'section_guess': _guess_section_from_text(text),
                    'confidence': 0.1  # низкая уверенность для простого текста
                })
        
        elif hasattr(element, 'name'):
            # Формируем путь к элементу для сохранения иерархии
            current_path = f"{parent_path}/{element.name}" if parent_path else element.name
            
            # Обрабатываем текст непосредственно в этом элементе
            direct_text = element.string if element.string else None
            if direct_text and direct_text.strip() and not direct_text.isspace():
                block_counter += 1
                block_id = f"block_{block_counter}"
                
                result['detailed_blocks'].append({
                    'block_id': block_id,
                    'text': direct_text.strip(),
                    'element_type': element.name,
                    'parent_path': current_path,
                    'depth': depth,
                    'position': block_counter,
                    'raw_html': str(element),
                    'section_guess': _guess_section_from_text(direct_text.strip()),
                    'element_attributes': _extract_element_attributes(element),
                    'confidence': _calculate_confidence_by_element(element.name)
                })
            
            # Обрабатываем все дочерние элементы
            for child in element.children:
                process_element(child, current_path, depth + 1)
    
    # Начинаем обработку с тела документа
    body = soup.body if soup.body else soup
    process_element(body)
    
    # Обновляем статистику
    result['processing_stats']['paragraphs_found'] = sum(1 for block in result['detailed_blocks'] 
                                                      if block['element_type'] in ['p', 'div', 'span'])
    result['processing_stats']['headers_found'] = sum(1 for block in result['detailed_blocks'] 
                                                    if block['element_type'] in ['h1', 'h2', 'h3', 'h4', 'h5', 'h6'])

def _extract_all_tables_completely(soup, result):
    """Извлекает ВСЕ таблицы без какой-либо фильтрации"""
    tables = soup.find_all('table')
    result['processing_stats']['tables_found'] = len(tables)
    
    for i, table in enumerate(tables):
        table_id = f"table_{i + 1}"
        
        # Извлекаем ВСЕ строки и ячейки без фильтрации
        rows = []
        for row_idx, tr in enumerate(table.find_all('tr')):
            cells = []
            for cell_idx, td in enumerate(tr.find_all(['td', 'th'])):
                cell_text = td.get_text(strip=False)  # НЕ обрезаем пробелы!
                cell_text = unescape(cell_text)
                cells.append({
                    'cell_id': f"{table_id}_cell_{row_idx}_{cell_idx}",
                    'raw_text': cell_text,
                    'clean_text': cell_text.strip() if cell_text.strip() else '[пустая ячейка]',
                    'attributes': _extract_element_attributes(td),
                    'row_span': td.get('rowspan', '1'),
                    'col_span': td.get('colspan', '1')
                })
            if cells:  # Добавляем строку даже если некоторые ячейки пустые
                rows.append({
                    'row_id': f"{table_id}_row_{row_idx}",
                    'cells': cells,
                    'attributes': _extract_element_attributes(tr)
                })
        
        # Извлекаем заголовки таблицы (если есть)
        headers = []
        caption = table.caption.get_text(strip=True) if table.caption else ''
        
        # Формируем полный текст таблицы
        table_full_text = f"ТАБЛИЦА {i + 1}\n"
        if caption:
            table_full_text += f"Заголовок: {caption}\n"
        
        # Добавляем все ячейки в detailed_blocks
        for row in rows:
            for cell in row['cells']:
                if cell['raw_text'].strip() or True:  # Добавляем ВСЕ ячейки, даже пустые
                    result['detailed_blocks'].append({
                        'block_id': f"table_cell_{cell['cell_id']}",
                        'text': cell['raw_text'],
                        'element_type': 'table_cell',
                        'parent_path': f"/table/{table_id}",
                        'depth': 3,
                        'position': len(result['detailed_blocks']) + 1,
                        'raw_html': str(cell),
                        'section_guess': 'lab_results' if any(word in cell['raw_text'].lower() 
                                                             for word in ['анализ', 'показатель', 'норма']) else 'other',
                        'table_metadata': {
                            'table_id': table_id,
                            'row_id': row['row_id'],
                            'cell_id': cell['cell_id'],
                            'caption': caption
                        },
                        'confidence': 0.8
                    })
        
        # Сохраняем полную структуру таблицы
        table_data = {
            'table_id': table_id,
            'caption': caption,
            'rows': rows,
            'raw_html': str(table)[:1000] + '...' if len(str(table)) > 1000 else str(table),
            'total_rows': len(rows),
            'total_cells': sum(len(row['cells']) for row in rows)
        }
        
        # Ищем в списке таблиц, если нет - добавляем
        table_exists = False
        for existing_table in result.get('tables', []):
            if existing_table['table_id'] == table_id:
                table_exists = True
                break
        
        if not table_exists:
            if 'tables' not in result:
                result['tables'] = []
            result['tables'].append(table_data)

def _extract_text_from_attributes(soup, result):
    """Извлекает текст из ВСЕХ атрибутов элементов"""
    attribute_counter = 0
    
    for element in soup.find_all(True):  # Все элементы
        attrs = element.attrs
        for attr_name, attr_value in attrs.items():
            if isinstance(attr_value, str) and len(attr_value.strip()) > 2:
                # Пропускаем технические атрибуты
                technical_attrs = ['class', 'id', 'style', 'height', 'width', 'border', 'cellpadding', 'cellspacing']
                if attr_name.lower() in technical_attrs:
                    continue
                
                # Пропускаем URL и очень короткие значения
                if attr_value.startswith(('http', 'www', '#', 'javascript:', 'data:')):
                    continue
                
                attribute_counter += 1
                result['detailed_blocks'].append({
                    'block_id': f"attr_{attribute_counter}",
                    'text': attr_value.strip(),
                    'element_type': 'attribute',
                    'parent_path': f"/{element.name}",
                    'depth': 1,
                    'position': len(result['detailed_blocks']) + 1,
                    'raw_html': f"{element.name}[{attr_name}]='{attr_value}'",
                    'section_guess': 'metadata',
                    'attribute_metadata': {
                        'element_name': element.name,
                        'attribute_name': attr_name,
                        'attribute_value': attr_value
                    },
                    'confidence': 0.6
                })

def _extract_html_comments(soup, result):
    """Извлекает ВСЕ HTML комментарии"""
    comments = soup.find_all(string=lambda text: isinstance(text, Comment))
    comment_counter = 0
    
    for comment in comments:
        comment_text = comment.strip()
        if comment_text and len(comment_text) > 2:
            comment_counter += 1
            result['detailed_blocks'].append({
                'block_id': f"comment_{comment_counter}",
                'text': comment_text,
                'element_type': 'comment',
                'parent_path': '/comment',
                'depth': 0,
                'position': len(result['detailed_blocks']) + 1,
                'raw_html': f"<!--{comment_text}-->",
                'section_guess': 'hidden_metadata',
                'confidence': 0.4
            })

def _extract_hidden_text(soup, result):
    """Извлекает скрытый текст (display:none, visibility:hidden)"""
    hidden_counter = 0
    
    # Все элементы с display:none
    hidden_elements = soup.find_all(style=re.compile('display\\s*:\\s*none', re.I))
    
    # Элементы с классами, содержащими hidden/invisible
    hidden_classes = soup.find_all(class_=re.compile('hidden|invisible|sr-only|visually-hidden', re.I))
    
    # Элементы с атрибутом hidden
    hidden_attrs = soup.find_all(attrs={'hidden': True})
    
    all_hidden = set(hidden_elements + hidden_classes + hidden_attrs)
    
    for element in all_hidden:
        hidden_text = element.get_text(strip=False)
        if hidden_text and hidden_text.strip() and len(hidden_text.strip()) > 5:
            hidden_counter += 1
            result['detailed_blocks'].append({
                'block_id': f"hidden_{hidden_counter}",
                'text': hidden_text.strip(),
                'element_type': 'hidden_element',
                'parent_path': f"/{element.name}",
                'depth': 2,
                'position': len(result['detailed_blocks']) + 1,
                'raw_html': str(element)[:200] + '...',
                'section_guess': 'hidden_data',
                'hidden_metadata': {
                    'element_name': element.name,
                    'hidden_reason': 'css_display_none' if 'display:none' in str(element) else 'class_or_attribute'
                },
                'confidence': 0.7
            })

def _build_element_hierarchy(soup, result):
    """Строит иерархию элементов для понимания структуры документа"""
    hierarchy = []
    
    def traverse(element, path="", level=0):
        if hasattr(element, 'name'):
            current_path = f"{path}/{element.name}" if path else element.name
            
            # Собираем информацию об элементе
            element_info = {
                'path': current_path,
                'tag': element.name,
                'level': level,
                'text_length': len(element.get_text(strip=False)),
                'has_children': len(list(element.children)) > 0,
                'attributes': _extract_element_attributes(element)
            }
            hierarchy.append(element_info)
            
            # Рекурсивно обходим детей
            for child in element.children:
                if hasattr(child, 'name') or isinstance(child, NavigableString):
                    traverse(child, current_path, level + 1)
    
    traverse(soup.body if soup.body else soup)
    result['element_hierarchy'] = hierarchy

def _extract_element_attributes(element):
    """Извлекает все атрибуты элемента в читаемом формате"""
    if not hasattr(element, 'attrs'):
        return {}
    
    attrs = {}
    for attr_name, attr_value in element.attrs.items():
        if isinstance(attr_value, (str, int, float, bool)):
            attrs[attr_name] = str(attr_value)
        elif isinstance(attr_value, list):
            attrs[attr_name] = " ".join(str(x) for x in attr_value)
    
    # Добавляем CSS классы отдельно для удобства
    if 'class' in attrs:
        attrs['css_classes'] = attrs['class'].split()
    
    return attrs

def _calculate_confidence_by_element(tag_name):
    """Рассчитывает уверенность в классификации на основе типа элемента"""
    high_confidence_tags = ['h1', 'h2', 'h3', 'strong', 'b', 'caption', 'th']
    medium_confidence_tags = ['h4', 'h5', 'h6', 'p', 'div', 'td', 'span']
    
    if tag_name in high_confidence_tags:
        return 0.9
    elif tag_name in medium_confidence_tags:
        return 0.7
    else:
        return 0.5

def _guess_section_from_text(text):
    """Улучшенное угадывание секции по тексту с большим количеством ключевых слов"""
    text_lower = text.lower()
    
    # Анамнез и жалобы
    if any(word in text_lower for word in ['жалоб', 'жалуется', 'обращается с', 'при поступлении', 'предъявляет']):
        return 'complaints'
    elif any(word in text_lower for word in ['анамнез', 'ранее болел', 'в прошлом', 'перенес', 'сопутствующ', 'аллерг', 'наследственность']):
        return 'history'
    
    # Диагностика
    elif any(word in text_lower for word in ['диагноз', 'ds-', 'мкб', 'поставлен диагноз', 'клинический диагноз', 'основной диагноз']):
        return 'diagnosis'
    elif any(word in text_lower for word in ['осмотр', 'состояние', 'объективный статус', 'физикальный', 'температура', 'пульс', 'давление', 'дыхание']):
        return 'physical_exam'
    
    # Исследования
    elif any(word in text_lower for word in ['анализы', 'показатели', 'результаты', 'лабораторные', 'биохимия', 'кровь', 'моча', 'тест']):
        return 'lab_results'
    elif any(word in text_lower for word in ['инструментальные', 'рентген', 'узи', 'мрт', 'кт', 'эхокг', 'эгдс', 'фгдс', 'ультразвук']):
        return 'instrumental'
    
    # Лечение
    elif any(word in text_lower for word in ['лечение', 'терапия', 'назначено', 'препарат', 'дозировка', 'схема', 'прием', 'курс', 'антибиотик']):
        return 'treatment'
    elif any(word in text_lower for word in ['медикаменты', 'лекарства', 'таблетки', 'уколы', 'раствор', 'капсулы', 'таб', 'мг']):
        return 'medications'
    
    # Заключение
    elif any(word in text_lower for word in ['заключение', 'эпикриз', 'итоги', 'резюме', 'суммируя', 'таким образом']):
        return 'conclusion'
    elif any(word in text_lower for word in ['рекомендации', 'план', 'наблюдение', 'диспансеризация', 'выписка', 'амбулаторно']):
        return 'recommendations'
    
    # Пациент
    elif any(word in text_lower for word in ['пациент', 'больной', 'фамилия', 'имя', 'отчество', 'возраст', 'пол', 'адрес', 'паспорт', 'полис']):
        return 'patient_info'
    
    return 'other'

def prepare_maximal_training_dataset(parsed_data):
    """
    Подготавливает МАКСИМАЛЬНО ПОЛНЫЙ датасет для обучения
    Каждый текстовый блок становится отдельным сэмплом
    """
    training_samples = []
    
    # Обрабатываем ВСЕ текстовые блоки
    for block in parsed_data['detailed_blocks']:
        text = block['text'].strip()
        if text:  # Добавляем ЛЮБОЙ непустой текст
            sample = {
                'text': text,
                'block_id': block['block_id'],
                'element_type': block['element_type'],
                'section_guess': block['section_guess'],
                'confidence': block['confidence'],
                'position': block['position'],
                'parent_path': block.get('parent_path', ''),
                'source_type': 'detailed_block'
            }
            
            # Добавляем дополнительные метаданные для контекста
            if 'table_metadata' in block:
                sample['source_type'] = 'table_cell'
                sample['table_info'] = block['table_metadata']
            elif 'attribute_metadata' in block:
                sample['source_type'] = 'attribute'
                sample['attribute_info'] = block['attribute_metadata']
            elif 'hidden_metadata' in block:
                sample['source_type'] = 'hidden_element'
                sample['hidden_info'] = block['hidden_metadata']
            
            training_samples.append(sample)
    
    # Добавляем полный текст документа как отдельные сэмплы (разбиваем на части)
    full_text = parsed_data['raw_full_text']
    if full_text and len(full_text) > 100:
        # Разбиваем полный текст на части по 500-1000 символов
        chunks = textwrap.wrap(full_text, width=800, break_long_words=False, replace_whitespace=False)
        for i, chunk in enumerate(chunks):
            if len(chunk.strip()) > 50:
                training_samples.append({
                    'text': chunk.strip(),
                    'block_id': f"full_text_chunk_{i}",
                    'element_type': 'full_text_chunk',
                    'section_guess': 'mixed_content',
                    'confidence': 0.2,
                    'position': len(training_samples) + 1,
                    'parent_path': '/full_document',
                    'source_type': 'full_text_chunk'
                })
    
    return training_samples

def process_single_doca_report_maximal(html_file_path, output_folder=None):
    """
    Обработка ОДНОГО отчета с МАКСИМАЛЬНЫМ извлечением данных
    """
    try:
        print(f"🚀 Начало обработки файла: {html_file_path}")
        
        # Читаем файл
        with open(html_file_path, 'rb') as f:
            html_content = f.read()
        
        # Максимально полный парсинг
        parsed_data = extract_all_text_with_structure(html_content)
        
        # Подготовка данных для обучения
        training_samples = prepare_maximal_training_dataset(parsed_data)
        
        print(f"✅ Успешно обработано!")
        print(f"📊 Статистика обработки:")
        print(f"   • Всего текстовых блоков: {len(parsed_data['detailed_blocks'])}")
        print(f"   • Всего сэмплов для обучения: {len(training_samples)}")
        print(f"   • Общая длина текста: {parsed_data['document_metadata']['total_text_length']} символов")
        print(f"   • Найдено таблиц: {parsed_data['processing_stats']['tables_found']}")
        
        # Сохранение результатов
        if output_folder:
            os.makedirs(output_folder, exist_ok=True)
            file_name = os.path.basename(html_file_path)
            base_name = os.path.splitext(file_name)[0]
            
            # Полные структурированные данные
            json_path = os.path.join(output_folder, f"{base_name}_full_structure.json")
            with open(json_path, 'w', encoding='utf-8') as f:
                json.dump(parsed_data, f, ensure_ascii=False, indent=2)
            
            # Датасет для обучения
            csv_path = os.path.join(output_folder, f"{base_name}_training_dataset.csv")
            pd.DataFrame(training_samples).to_csv(csv_path, index=False, encoding='utf-8')
            
            # Человекочитаемый полный текст
            txt_path = os.path.join(output_folder, f"{base_name}_full_text.txt")
            with open(txt_path, 'w', encoding='utf-8') as f:
                f.write(parsed_data['raw_full_text'])
            
            # Отчет о структуре
            report_path = os.path.join(output_folder, f"{base_name}_structure_report.txt")
            _generate_detailed_structure_report(parsed_data, report_path)
            
            print(f"💾 Результаты сохранены в папке: {output_folder}")
            print(f"   • Полная структура: {json_path}")
            print(f"   • Датасет для обучения: {csv_path}")
            print(f"   • Полный текст: {txt_path}")
            print(f"   • Отчет о структуре: {report_path}")
        
        return parsed_data, training_samples
    
    except Exception as e:
        print(f"❌ КРИТИЧЕСКАЯ ОШИБКА при обработке {html_file_path}: {str(e)}")
        import traceback
        traceback.print_exc()
        return None, None

def _generate_detailed_structure_report(parsed_data, report_path):
    """Генерирует подробный отчет о структуре документа"""
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write("=" * 80 + "\n")
        f.write("ПОДРОБНЫЙ ОТЧЕТ О СТРУКТУРЕ ДОКУМЕНТА\n")
        f.write("=" * 80 + "\n\n")
        
        f.write("📄 ОСНОВНАЯ ИНФОРМАЦИЯ:\n")
        f.write(f"   • Название документа: {parsed_data['document_metadata']['title']}\n")
        f.write(f"   • Общая длина текста: {parsed_data['document_metadata']['total_text_length']} символов\n")
        f.write(f"   • Кодировка: {parsed_data['document_metadata']['encoding_used']}\n")
        f.write(f"   • Парсер: {parsed_data['document_metadata']['parser_used']}\n\n")
        
        f.write("📊 СТАТИСТИКА ЭЛЕМЕНТОВ:\n")
        stats = parsed_data['processing_stats']
        f.write(f"   • Заголовков найдено: {stats['headers_found']}\n")
        f.write(f"   • Параграфов/блоков: {stats['paragraphs_found']}\n")
        f.write(f"   • Таблиц: {stats['tables_found']}\n")
        f.write(f"   • Списков: {stats['lists_found']}\n")
        f.write(f"   • Других элементов: {stats['other_elements_found']}\n")
        f.write(f"   • Создано текстовых блоков: {stats['text_blocks_created']}\n\n")
        
        f.write("🔍 РАСПРЕДЕЛЕНИЕ ПО СЕКЦИЯМ:\n")
        section_counts = defaultdict(int)
        for block in parsed_data['detailed_blocks']:
            section_counts[block['section_guess']] += 1
        
        for section, count in sorted(section_counts.items(), key=lambda x: x[1], reverse=True):
            f.write(f"   • {section}: {count} блоков\n")
        f.write("\n")
        
        f.write("🏗️ ИЕРАРХИЯ ЭЛЕМЕНТОВ (первые 20 уровней):\n")
        for i, element in enumerate(parsed_data['element_hierarchy'][:20]):
            indent = "  " * element['level']
            f.write(f"{indent}{element['path']} (уровень {element['level']}, текст: {element['text_length']} символов)\n")
        f.write("\n")
        
        f.write("🧩 ПРИМЕРЫ ТЕКСТОВЫХ БЛОКОВ (первые 10):\n")
        for i, block in enumerate(parsed_data['detailed_blocks'][:10]):
            f.write(f"\nБЛОК #{i + 1} (ID: {block['block_id']}):\n")
            f.write(f"   Тип элемента: {block['element_type']}\n")
            f.write(f"   Предполагаемая секция: {block['section_guess']}\n")
            f.write(f"   Уверенность: {block['confidence']:.2f}\n")
            f.write(f"   Путь: {block['parent_path']}\n")
            f.write(f"   Текст (первые 100 символов):\n")
            f.write(f"      {block['text'][:100] + '...' if len(block['text']) > 100 else block['text']}\n")

# Пример использования для одного файла
if __name__ == "__main__":
    # Укажите путь к вашему HTML файлу
    HTML_FILE_PATH = "./АбашкинЮН_history_0.html"  # ЗАМЕНИТЕ НА ВАШ ПУТЬ
    OUTPUT_FOLDER = "."  # ЗАМЕНИТЕ НА ВАШ ПУТЬ
    
    print("🔥 ЗАПУСК МАКСИМАЛЬНО ПОЛНОГО ПАРСЕРА DOCA+")
    print("=" * 60)
    
    # Обработка одного файла
    parsed_data, training_samples = process_single_doca_report_maximal(
        HTML_FILE_PATH, 
        OUTPUT_FOLDER
    )
    
    if parsed_data and training_samples:
        print(f"\n🎉 ОБРАБОТКА УСПЕШНО ЗАВЕРШЕНА!")
        print(f"\n💡 СЛЕДУЮЩИЕ ШАГИ ДЛЯ ОБУЧЕНИЯ МОДЕЛЕЙ:")
        print("""
# Загрузка датасета
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Чтение датасета
df = pd.read_csv('ваш_файл_training_dataset.csv')

# Подготовка данных
X = df['text']
y = df['section_guess']  # или другая целевая переменная

# Векторизация - используем ВСЕ данные
vectorizer = TfidfVectorizer(
    max_features=10000,  # много фичей для максимального охвата
    ngram_range=(1, 3),   # unigrams, bigrams, trigrams
    min_df=2,            # игнорируем очень редкие слова
    max_df=0.95          # игнорируем очень частые слова
)
X_vec = vectorizer.fit_transform(X)

# Обучение логистической регрессии
model = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',  # балансировка классов
    random_state=42,
    solver='lbfgs'           # эффективный солвер для больших данных
)
model.fit(X_vec, y)

# Оценка качества
from sklearn.metrics import classification_report
y_pred = model.predict(X_vec)
print(classification_report(y, y_pred))

# Сохранение моделей
import joblib
joblib.dump(model, 'doca_full_classifier.pkl')
joblib.dump(vectorizer, 'doca_text_vectorizer.pkl')
        """)
    
    else:
        print("\n❌ ОБРАБОТКА НЕ УДАЛАСЬ. Проверьте путь к файлу и права доступа.")


🔥 ЗАПУСК МАКСИМАЛЬНО ПОЛНОГО ПАРСЕРА DOCA+
🚀 Начало обработки файла: ./АбашкинЮН_history_0.html
❌ КРИТИЧЕСКАЯ ОШИБКА при обработке ./АбашкинЮН_history_0.html: 'NavigableString' object has no attribute 'children'

❌ ОБРАБОТКА НЕ УДАЛАСЬ. Проверьте путь к файлу и права доступа.


Traceback (most recent call last):
  File "/tmp/ipykernel_3201291/109542369.py", line 494, in process_single_doca_report_maximal
    parsed_data = extract_all_text_with_structure(html_content)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_3201291/109542369.py", line 81, in extract_all_text_with_structure
    _build_element_hierarchy(soup, result)
  File "/tmp/ipykernel_3201291/109542369.py", line 354, in _build_element_hierarchy
    traverse(soup.body if soup.body else soup)
  File "/tmp/ipykernel_3201291/109542369.py", line 352, in traverse
    traverse(child, current_path, level + 1)
  File "/tmp/ipykernel_3201291/109542369.py", line 344, in traverse
    'has_children': len(list(element.children)) > 0,
                             ^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/bs4/element.py", line 984, in __getattr__
    raise AttributeError(
AttributeError: 'NavigableString' object has no attribute 'children'
